# Churn Pipeline — Exploration Notebook

This notebook walks through the essence of [05-ml-pipeline-churn](/projects/05-ml-pipeline-churn):
churn as a **cost-minimization** problem (one false negative — a lost customer —
costs 5× one false positive — a retention voucher). It demonstrates, in a few
seconds and without retraining anything:

1. What the prepared data looks like (from the persisted DVC artifacts in `data/`).
2. The headline comparison — the five-run evaluation ladder from `reports/metrics.json`.
3. A reproduction of the key result: the naive-vs-selected-model business cost
   delta, recomputed live from the persisted model artifact and holdout frame.

Everything below reads persisted artifacts. Data comes from `dvc repro`; nothing
is re-downloaded or retrained.

## How to run

From the project directory (`projects/05-ml-pipeline-churn`):

```bash
uv sync --all-packages   # from the repo root, once
uv run jupyter notebook notebooks/exploration.ipynb
```

Then run cells top to bottom. The artifacts in `data/` (parquet frames, pickled
pipelines, metrics) are produced by `dvc repro`; if they are missing, run
`uv run dvc repro` first.

## 1. Load the prepared data

Artifact locations come from the package's own config (`ChurnConfig` +
`artifact_paths`), so this notebook can never drift from where the pipeline
actually writes. The holdout frame is the same 1,000-row stratified holdout
used for every number in the case study — never seen by HPO, fitting, or calibration.

In [ ]:
import pandas as pd
from churn_pipeline.config import ChurnConfig
from churn_pipeline.evaluate import artifact_paths

cfg = ChurnConfig()
paths = artifact_paths(cfg)

holdout = pd.read_parquet(paths["holdout"])
X_hold, y_hold = holdout.drop(columns=["churned"]), holdout["churned"].astype(int)

print(
    f"holdout: {X_hold.shape[0]} rows x {X_hold.shape[1]} features, base rate {y_hold.mean():.3f}"
)
X_hold.head(3)

## 2. The headline comparison — the run ladder

Every candidate was scored through the same decision procedure: out-of-fold-free
holdout probabilities, a 41-point threshold grid, and business cost
`FP·1 + FN·5` at each run's operating point. The ladder below is read straight
from the persisted `reports/metrics.json` (written by `dvc repro`).

In [ ]:
import json

with open(cfg.reports_dir / "metrics.json") as f:
    metrics = json.load(f)

ci = metrics["_meta"]["cost_delta_ci"]
ladder = pd.DataFrame(metrics).drop(columns=["_meta"]).T
cols = ["business_cost", "f1_macro", "roc_auc", "ece", "accuracy"]
ladder[cols].round(4)

Two things to notice in that table:

- **Cost vs naive: 440 → 282 (−35.9%)** for the selected cost-HPO logistic at its
  cost-optimal threshold (0.700) — and the paired bootstrap CI on the delta is
  significant: **−158, 95% CI [−237, −87]** (`n_boot=2,000`, seeded).
- **The naive run has the best accuracy (0.912)** — on an 8.8% churn rate, never
  alarming is accurate and useless. That is why this project is judged in money,
  not accuracy.

In [ ]:
print(
    f"cost delta vs naive: {ci['point']:.0f}  "
    f"95% CI [{ci['lo']:.0f}, {ci['hi']:.0f}]  "
    f"(significant: {ci['significant_95']}, {ci['n_boot']} resamples)"
)

## 3. Reproduce the key result live

Recompute the naive-vs-model cost from the persisted artifacts: score the raw
holdout rows through the shipped pipeline (`predict_proba` — preprocessing is
embedded, so raw rows go in), sweep the same 41-point threshold grid with the
package's own `business_cost`, and compare against the naive majority run at
the training base rate. This takes seconds and should exactly match
`reports/metrics.json`.

In [ ]:
import joblib
import numpy as np
from churn_pipeline.evaluate import business_cost, naive_majority_run

model = joblib.load(paths["selected"])
with open(cfg.data_dir / "model_meta.json") as f:
    train_base_rate = json.load(f)["train_churn_rate"]

proba = model.predict_proba(X_hold)[:, 1]
y_true = y_hold.to_numpy()

model_costs = [
    business_cost(y_true, proba, t, cfg.fp_cost, cfg.fn_cost) for t in cfg.threshold_grid
]
model_opt = cfg.threshold_grid[int(np.argmin(model_costs))]

_, _, naive_opt = naive_majority_run(cfg, y_true, train_base_rate)
naive_proba = np.full(len(y_true), train_base_rate)
naive_cost = business_cost(y_true, naive_proba, naive_opt, cfg.fp_cost, cfg.fn_cost)

savings = naive_cost - min(model_costs)
print(f"selected model: cost {min(model_costs):.0f} at threshold {model_opt:.3f}")
print(f"naive majority: cost {naive_cost:.0f} at threshold {naive_opt:.3f}")
print(f"savings vs naive: {savings:.0f} cost units ({100 * savings / naive_cost:.1f}%)")

assert min(model_costs) == metrics["selected_model"]["business_cost"]
assert model_opt == metrics["_meta"]["selected_opt"]
assert naive_cost == metrics["naive_majority"]["business_cost"]
print("matches reports/metrics.json — reproduction verified")

## Where to go next

- [Case study: 05-ml-pipeline-churn](/projects/05-ml-pipeline-churn) — the full
  narrative: data contract, cost model, selection, and the decision-curve analysis.
- [DESIGN.md](../DESIGN.md) — architecture decisions: pipeline-embedded
  preprocessing, cost-aware Optuna objective, threshold-as-business-decision.
- Code tree: `src/churn_pipeline/` — `config.py` (single source of truth),
  `features.py` (OutlierCapper / MissingValueIndicator), `hpo.py` (Optuna study),
  `evaluate.py` (cost curve, bootstrap CI, figures), `cli.py` (`prepare` / `train` / `evaluate`).

To regenerate every artifact from scratch: `uv run dvc repro`.